# 02 - Rightsizing Engine Development & Debug

Run the rightsizing engine **locally** using Athena to load reference data
(instead of Spark/Glue). This lets you:

- Step through `get_recommendation()` with real data
- Test pricing resolution chain
- Debug SKU catalog lookups
- Validate rule matching logic
- Test with custom resource_data and metrics

In [ ]:
import sys, os
import pandas as pd

# Add paths
nb_dir = os.path.dirname(os.path.abspath('__file__'))
project_root = os.path.join(nb_dir, '..')
scripts_common = os.path.join(project_root, 'src', 's3', 'scripts', 'common')

sys.path.insert(0, nb_dir)
sys.path.insert(0, scripts_common)

from local_helpers import LocalIcebergReader, LocalConfigLoader, athena_query

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 100)

## 1. Load Reference Data via Athena

In [ ]:
CLOUD = 'aws'  # Change to 'azure', 'gcp', etc.

reader = LocalIcebergReader()

print('Loading reference data from Athena...')
catalog_pdf = reader.load_sku_catalog(CLOUD)
print(f'  SKU catalog:      {len(catalog_pdf)} rows')

pricing_pdf = reader.load_pricing_history(CLOUD)
print(f'  Pricing history:  {len(pricing_pdf)} rows')

configs_pdf = reader.load_service_configs(CLOUD)
print(f'  Service configs:  {len(configs_pdf)} rows')

rules_pdf = reader.load_rules(CLOUD)
print(f'  Rightsizing rules: {len(rules_pdf)} rows')

print('\nDone.')

## 2. Load JSON Configs (local files or S3)

In [ ]:
config_loader = LocalConfigLoader()

provider_config = config_loader.load_json('provider_config.json', {})
print(f'provider_config keys: {list(provider_config.keys())}')

cur_mappings = config_loader.load_json('cur_mappings.json', {})
print(f'cur_mappings keys: {list(cur_mappings.keys())}')

sizing_defaults = config_loader.load_json('sizing_defaults.json', {})
print(f'sizing_defaults keys: {list(sizing_defaults.keys())}')

## 3. Initialize Engine Components (without Spark)

We instantiate each engine component individually using the Athena-loaded
DataFrames and local config. This gives you access to each component for
step-by-step debugging.

In [ ]:
from rightsizing_engine.constants import load_mappings, SIZE_ORDINAL, AZURE_TIER_ORDINAL
from rightsizing_engine.models import RightSizingResult, RightsizingRule, ServiceConfig
from rightsizing_engine.sku_catalog import SKUCatalog
from rightsizing_engine.pricing_resolver import PricingResolver
from rightsizing_engine.service_config_loader import ServiceConfigLoader
from rightsizing_engine.sizing_strategies import SizingStrategyFactory

# Load mappings from local config (no S3ConfigLoader needed)
mappings = load_mappings()  # uses embedded defaults

# Override with JSON config if loaded
if sizing_defaults:
    mappings['size_ordinal'] = sizing_defaults.get('size_ordinal', SIZE_ORDINAL)
    mappings['azure_tier_ordinal'] = sizing_defaults.get('azure_tier_ordinal', AZURE_TIER_ORDINAL)
    mappings['recommendation_thresholds'] = sizing_defaults.get('recommendation_thresholds', {})
    mappings['storage_migration_maps'] = sizing_defaults.get('storage_migration_maps', {})
    mappings['severity_levels'] = sizing_defaults.get('severity_levels', {})

# Inject sizing defaults into provider_config
provider_config['_sizing_defaults'] = {
    'size_ordinal': mappings.get('size_ordinal', {}),
    'azure_tier_ordinal': mappings.get('azure_tier_ordinal', {}),
}

print('Mappings loaded. Thresholds:', mappings.get('recommendation_thresholds', {}))

In [ ]:
# Initialize core components
sku_catalog = SKUCatalog(
    catalog_pdf,
    tier_ordinal=mappings.get('azure_tier_ordinal'),
)
print(f'SKUCatalog initialized: {len(catalog_pdf)} SKUs')

pricing_resolver = PricingResolver(
    pricing_pdf, sku_catalog,
    provider_config=provider_config,
    cloud_provider=CLOUD,
)
print(f'PricingResolver initialized')

config_loader_svc = ServiceConfigLoader(configs_pdf)
service_configs = config_loader_svc.load_configs(CLOUD)
print(f'ServiceConfigs loaded: {len(service_configs)} services')

strategy_factory = SizingStrategyFactory()
print(f'SizingStrategyFactory initialized')

# Parse rules
rules = []
if rules_pdf is not None and len(rules_pdf) > 0:
    for _, row in rules_pdf.iterrows():
        try:
            rules.append(RightsizingRule.from_row(row.to_dict()))
        except Exception as e:
            print(f'  Rule parse error: {e}')
    rules.sort(key=lambda r: r.priority)
print(f'Rules parsed: {len(rules)}')

## 4. Test Price Lookups

In [ ]:
# Test pricing resolver
test_skus = ['m5.large', 'm5.xlarge', 'm5.2xlarge', 't3.micro', 'r5.large']

for sku in test_skus:
    price, source = pricing_resolver.get_effective_price(
        CLOUD, 'Amazon EC2', sku, 'us-east-1'
    )
    print(f'  {sku:20s} -> ${price:.4f}/hr  (source: {source})')

## 5. Test SKU Catalog Lookups

In [ ]:
# Look up SKU info
sku_info = sku_catalog.get_sku_info('Amazon EC2', 'm5.xlarge', 'us-east-1')
if sku_info:
    print(f'SKU: {sku_info}')
else:
    # Try direct DataFrame lookup
    match = catalog_pdf[
        (catalog_pdf['sku_code'] == 'm5.xlarge') & 
        (catalog_pdf['service_name'] == 'Amazon EC2')
    ]
    print(f'Direct lookup: {len(match)} matches')
    display(match)

In [ ]:
# Find smaller SKUs in the same family
family_skus = catalog_pdf[
    (catalog_pdf['sku_code'].str.startswith('m5.')) &
    (catalog_pdf['service_name'] == 'Amazon EC2') &
    (catalog_pdf['region'] == 'us-east-1')
].sort_values('hourly_price')

display(family_skus[['sku_code', 'vcpu', 'memory_gb', 'hourly_price']])

## 6. Test Single Recommendation (step-by-step)

In [ ]:
# Define test resource
resource_data = {
    'resource_id': 'i-test-001',
    'service_name': 'Amazon EC2',
    'service_category': 'Compute',
    'current_sku': 'm5.xlarge',
    'region': 'us-east-1',
    'account_id': '123456789012',
}

metrics = {
    'cpu_avg': 8.5,
    'memory_avg': 22.0,
    'network_in_avg': 100.0,
    'network_out_avg': 50.0,
}

current_cost = 140.16  # Monthly from CUR

print('Resource:', resource_data)
print('Metrics:', metrics)
print('Current cost: $', current_cost)

In [ ]:
# Step 1: Find service config
svc_config = service_configs.get(resource_data['service_name'])
print(f'Service config found: {svc_config is not None}')
if svc_config:
    print(f'  Category: {svc_config.service_category}')
    print(f'  Primary metric: {svc_config.primary_metric}')

In [ ]:
# Step 2: Find applicable rules
service_category = resource_data.get('service_category', '')
service_name = resource_data.get('service_name', '')

applicable_rules = [
    r for r in rules
    if r.service_category == service_category
    and (r.service_name is None or r.service_name == service_name)
]

print(f'Applicable rules: {len(applicable_rules)}')
for r in applicable_rules:
    print(f'  [{r.priority}] {r.rule_code}: {r.sizing_method} -> {r.action_type}')

In [ ]:
# Step 3: Try each rule
for rule in applicable_rules:
    print(f'\n--- Rule: {rule.rule_code} ({rule.sizing_method}) ---')
    try:
        strategy = strategy_factory.get_strategy(rule.sizing_method)
    except ValueError as e:
        print(f'  Strategy not found: {e}')
        continue
    
    effective_params = dict(rule.sizing_params)
    recommended_sku = strategy.recommend(
        resource_data, metrics, sku_catalog, svc_config, effective_params
    )
    
    print(f'  Recommended SKU: {recommended_sku}')
    
    if recommended_sku and recommended_sku != resource_data['current_sku']:
        # Get prices
        cur_price, cur_src = pricing_resolver.get_effective_price(
            CLOUD, service_name, resource_data['current_sku'], resource_data['region']
        )
        rec_price, rec_src = pricing_resolver.get_effective_price(
            CLOUD, service_name, recommended_sku, resource_data['region']
        )
        savings = pricing_resolver.calculate_savings(cur_price, rec_price)
        
        print(f'  Current price:  ${cur_price:.4f}/hr ({cur_src})')
        print(f'  New price:      ${rec_price:.4f}/hr ({rec_src})')
        print(f'  Monthly savings: ${savings["monthly_savings"]:.2f}')
        print(f'  Annual savings:  ${savings["annual_savings"]:.2f}')
        print(f'  Savings %:       {savings["savings_percentage"]:.1f}%')
        break  # First successful rule wins

## 7. Batch Test with Bronze Data

Pull real EC2 instances from bronze tables and run the engine over them.

In [ ]:
# Load real EC2 instances from bronze layer
ec2_df = athena_query("""
    SELECT 
        instance_id as resource_id,
        instance_type as current_sku,
        region,
        account_id
    FROM bronze_aws_ec2_instances
    LIMIT 50
""")
print(f'Loaded {len(ec2_df)} EC2 instances')
ec2_df.head()

In [ ]:
# Load CloudWatch metrics for these instances
if not ec2_df.empty:
    instance_ids = "','".join(ec2_df['resource_id'].tolist()[:20])
    metrics_df = athena_query(f"""
        SELECT 
            resource_id,
            AVG(CASE WHEN metric_name = 'CPUUtilization' THEN average END) as cpu_avg,
            AVG(CASE WHEN metric_name = 'NetworkIn' THEN average END) as network_in_avg,
            AVG(CASE WHEN metric_name = 'NetworkOut' THEN average END) as network_out_avg
        FROM bronze_aws_cloudwatch_metrics
        WHERE resource_id IN ('{instance_ids}')
        GROUP BY resource_id
    """)
    print(f'Loaded metrics for {len(metrics_df)} instances')
    display(metrics_df.head())

In [ ]:
# Generate recommendations for each instance
from rightsizing_engine.constants import HOURS_PER_MONTH, Confidence

results = []
if not ec2_df.empty and not metrics_df.empty:
    merged = ec2_df.merge(metrics_df, on='resource_id', how='left')
    
    for _, row in merged.iterrows():
        rd = {
            'resource_id': row['resource_id'],
            'service_name': 'Amazon EC2',
            'service_category': 'Compute',
            'current_sku': row['current_sku'],
            'region': row['region'],
            'account_id': str(row.get('account_id', '')),
        }
        m = {
            'cpu_avg': float(row.get('cpu_avg', 0) or 0),
            'network_in_avg': float(row.get('network_in_avg', 0) or 0),
            'network_out_avg': float(row.get('network_out_avg', 0) or 0),
        }
        
        # Find recommendation (reusing components from above)
        svc_cfg = service_configs.get('Amazon EC2')
        if not svc_cfg:
            continue
        
        for rule in applicable_rules:
            try:
                strategy = strategy_factory.get_strategy(rule.sizing_method)
                rec_sku = strategy.recommend(rd, m, sku_catalog, svc_cfg, dict(rule.sizing_params))
                if rec_sku and rec_sku != rd['current_sku']:
                    cur_p, _ = pricing_resolver.get_effective_price(CLOUD, 'Amazon EC2', rd['current_sku'], rd['region'])
                    rec_p, _ = pricing_resolver.get_effective_price(CLOUD, 'Amazon EC2', rec_sku, rd['region'])
                    savings = pricing_resolver.calculate_savings(cur_p, rec_p)
                    results.append({
                        'resource_id': rd['resource_id'],
                        'current_sku': rd['current_sku'],
                        'recommended_sku': rec_sku,
                        'cpu_avg': m['cpu_avg'],
                        'monthly_savings': savings['monthly_savings'],
                        'annual_savings': savings['annual_savings'],
                        'rule': rule.rule_code,
                    })
                    break
            except Exception:
                continue

results_df = pd.DataFrame(results)
print(f'Generated {len(results_df)} recommendations')
if not results_df.empty:
    print(f'Total annual savings: ${results_df["annual_savings"].sum():,.2f}')
    display(results_df)

## 8. Sandbox: Edit & Re-test

Modify engine code in `src/s3/scripts/common/rightsizing_engine/`, then
reload here to test changes without redeploying.

In [ ]:
# Hot-reload engine modules after code changes
import importlib
import rightsizing_engine.sizing_strategies as ss
import rightsizing_engine.pricing_resolver as pr
import rightsizing_engine.sku_catalog as sc

importlib.reload(ss)
importlib.reload(pr)
importlib.reload(sc)

# Re-initialize with reloaded modules
sku_catalog = sc.SKUCatalog(catalog_pdf, tier_ordinal=mappings.get('azure_tier_ordinal'))
pricing_resolver = pr.PricingResolver(pricing_pdf, sku_catalog, provider_config=provider_config, cloud_provider=CLOUD)
strategy_factory = ss.SizingStrategyFactory()

print('Modules reloaded. Ready for re-testing.')